In [7]:
import sys
import pandas as pd
import numpy as np
import dagshub
import mlflow
import statsmodels.api as sm

# 1. Point to your scripts folder
sys.path.append("../../scripts")
import ml_utils as mlutils

# 2. Setup DagsHub
REPO_OWNER = "SamuelFredricBerg"
REPO_NAME = "4dt907"
MODEL_NAME = "Project_Model_A2_V2"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
utils = mlutils.mlutils(MODEL_NAME)

# 3. Recreate Assignment 2 Data (X and y)
def add_engineered_features(df):
    """Matches the exact logic used during your 0.79 R2 training."""
    # Example Symmetry logic - Replace with your actual column names!
    if 'LeftArm' in df.columns and 'RightArm' in df.columns:
        df['Arm_Symmetry'] = np.abs(df['LeftArm'] - df['RightArm'])
    
    if 'LeftKnee' in df.columns and 'RightKnee' in df.columns:
        df['Knee_Symmetry'] = np.abs(df['LeftKnee'] - df['RightKnee'])
        
    # Add any other ratios or averages you created
    return df

def remove_outliers_cooks(X, y):
    """Matches your training logic exactly."""
    X_const = sm.add_constant(X)
    influence = sm.OLS(y, X_const).fit().get_influence()
    cooks_d = influence.cooks_distance[0]
    # Uses the same threshold logic: 3 / len(X)
    threshold = 3 / len(X)
    mask = cooks_d < threshold
    return mask

def get_a2_data():
    data_path = "../../data/AimoScore_WeakLink_big_scores_A2.csv"
    df = pd.read_csv(data_path).drop_duplicates()
    
    # 1. Feature Engineering (Symmetries, etc.)
    df = add_engineered_features(df)
    
    # 2. Initial X/y split
    target_col = "AimoScore"
    y = df[target_col]
    cheating_cols = ["No_1_Time_Deviation", "No_2_Time_Deviation", "EstimatedScore", "ID", "Date"]
    X = df.drop(columns=[target_col] + cheating_cols, errors="ignore")
    X = X.select_dtypes(include=[np.number])

    # 3. CRITICAL: Remove Outliers (This is the missing 0.10 R2)
    mask = remove_outliers_cooks(X, y)
    X_clean = X[mask]
    y_clean = y[mask]
    
    print(f"✅ Data Recreated & Cleaned: {X_clean.shape[0]} samples remaining.")
    return X_clean, y_clean

# 4. Run the 5-Fold R2 Test
X, y = get_a2_data()
print(f"Validating models on {X.shape[0]} samples with {X.shape[1]} features...")

# Perform the registry check
registry_results = utils.compare_all_registry_aliases_r2(X, y, n_splits=5)



Initialized MLflow to track repo "SamuelFredricBerg/4dt907"

Repository SamuelFredricBerg/4dt907 initialized!

✅ Data Recreated & Cleaned: 1895 samples remaining.
Validating models on 1895 samples with 38 features...

--- Registry Validation: Comparing ['prod', 'dev', 'backup'] (5-Fold R2) ---


Testing @prod (Version 11)...


Testing @dev (Version 8)...


Testing @backup (Version 9)...

ALIAS      | MEAN R2    | STD DEV   
--------------------------------------------------
prod       | 0.7798     | 0.0102
dev        | 0.7694     | 0.0101
backup     | 0.7694     | 0.0101

